## Invoice DocAI --- 07 Literature Context & Why Models Differ

This notebook positions our experimental results within the research landscape
and provides a theoretical framework explaining **why** models behave differently.

### Plan
1. SROIE leaderboard comparison chart (our results vs published SOTA)
2. Architecture comparison: OCR pipeline vs end-to-end Donut
3. Why each field behaves differently: theoretical analysis
4. Error propagation framework
5. Gap analysis: why our scores are lower than published
6. Robustness: theoretical explanation
7. References

In [ ]:
# =============================================================================
# Cell 1 --- Setup
# =============================================================================
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.patches import Patch

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception:
    pass


def is_project_root(p: Path) -> bool:
    return (
        (p / 'data' / 'sroie' / 'processed' / 'manifest_val.csv').exists()
        and (p / 'v2' / 'src').exists()
    )


def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [
        cwd, cwd.parent, cwd.parent.parent, cwd.parent.parent.parent,
        Path('/content/invoice_docai'),
        Path('/content/drive/MyDrive/invoice_docai'),
        Path('/content/drive/MyDrive/ML Neimark/From OCR 2 Transformers/invoice_docai'),
        Path('/content/drive/MyDrive/Yandex.Disk/ML Neimark/From OCR 2 Transformers/invoice_docai'),
        Path('/content/drive/MyDrive/Yandex.Disk/Yandex.Disk/ML Neimark/From OCR 2 Transformers/invoice_docai'),
        Path(r'c:\\Yandex.Disk\\Yandex.Disk\\ML Neimark\\From OCR 2 Transformers\\invoice_docai'),
    ]
    for p in candidates:
        try:
            p = p.resolve()
        except Exception:
            continue
        if is_project_root(p):
            return p
    raise FileNotFoundError(f'Cannot locate project root. cwd={cwd}')


PROJECT_ROOT = resolve_project_root()
OUTPUT_ROOT  = PROJECT_ROOT / 'v2' / 'outputs'

mpl.rcParams.update({
    'figure.dpi': 110,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
})

print(f'PROJECT_ROOT = {PROJECT_ROOT}')

In [ ]:
# =============================================================================
# Cell 2 --- SROIE Leaderboard Comparison Chart
# =============================================================================
leaderboard = pd.DataFrame([
    {'Model': 'Our OCR Baseline\n(quick mode)',     'F1': 64.59,  'Type': 'OCR-based',  'Ours': True},
    {'Model': 'Our Donut-FT\n(quick mode)',         'F1': 74.87,  'Type': 'End-to-end', 'Ours': True},
    {'Model': 'BERT + SPADE\n(Hong et al. 2022)',   'F1': 93.67,  'Type': 'OCR-based',  'Ours': False},
    {'Model': 'Donut FT\n(Kim et al. 2022)',        'F1': 94.40,  'Type': 'End-to-end', 'Ours': False},
    {'Model': 'LayoutLM BASE\n(Xu et al. 2020)',    'F1': 95.11,  'Type': 'OCR-based',  'Ours': False},
    {'Model': 'PICK (GCN)\n(Yu et al. 2021)',       'F1': 96.10,  'Type': 'OCR-based',  'Ours': False},
    {'Model': 'LayoutLMv2 LARGE\n(Xu et al. 2021)', 'F1': 96.39,  'Type': 'OCR-based',  'Ours': False},
    {'Model': 'BROS LARGE\n(Hong et al. 2022)',     'F1': 96.62,  'Type': 'OCR-based',  'Ours': False},
])
leaderboard = leaderboard.sort_values('F1', ascending=True).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(11, 6))

colors = []
for _, row in leaderboard.iterrows():
    if row['Ours']:
        colors.append('#e74c3c' if row['Type'] == 'End-to-end' else '#3498db')
    else:
        colors.append('#f58518' if row['Type'] == 'End-to-end' else '#4c78a8')

bars = ax.barh(leaderboard['Model'], leaderboard['F1'],
               color=colors, edgecolor='black', linewidth=0.6, height=0.6)

# Bold outline for our results
for bar, is_ours in zip(bars, leaderboard['Ours']):
    if is_ours:
        bar.set_linewidth(2.5)
        bar.set_edgecolor('#c0392b')

# Value labels
for bar, val in zip(bars, leaderboard['F1']):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}', va='center', fontsize=10, fontweight='bold')

ax.set_xlabel('Entity-level F1 (%)', fontsize=12)
ax.set_title('SROIE 2019 Leaderboard: Our Results vs Published SOTA', fontsize=14)
ax.set_xlim(0, 108)

# Competitive zone line
ax.axvline(x=93, color='grey', linestyle='--', alpha=0.4)
ax.text(93.5, -0.3, 'Competitive\nzone', fontsize=8, color='grey', style='italic')

# Legend
legend_elements = [
    Patch(facecolor='#4c78a8', edgecolor='black', label='Published: OCR-based'),
    Patch(facecolor='#f58518', edgecolor='black', label='Published: End-to-end'),
    Patch(facecolor='#3498db', edgecolor='#c0392b', linewidth=2, label='Ours: OCR-based'),
    Patch(facecolor='#e74c3c', edgecolor='#c0392b', linewidth=2, label='Ours: End-to-end'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9)
ax.grid(axis='x', alpha=0.2)

plt.tight_layout()
plt.savefig(OUTPUT_ROOT / 'fig_sroie_leaderboard_quick.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_sroie_leaderboard_quick.png')

## Architecture Comparison

### Pipeline 1: OCR + Rule-Based Extraction

```
 Receipt Image
      \u2502
      \u25bc
 \u250c\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2510
 \u2502  Text Detection (CRAFT CNN)       \u2502 \u2190 Stage 1: WHERE is text?
 \u2514\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2518
      \u2502
      \u25bc
 \u250c\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2510
 \u2502  Text Recognition (CRNN + CTC)    \u2502 \u2190 Stage 2: WHAT does it say?
 \u2514\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2518
      \u2502
      \u25bc
 \u250c\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2510
 \u2502  Rule-based NER (regex + rules)   \u2502 \u2190 Stage 3: WHAT FIELD is it?
 \u2514\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2518
      \u2502
      \u25bc
 { vendor, date, total }
```

**Error propagation:** `P(correct) = P(detect) \u00d7 P(recognize|detect) \u00d7 P(extract|recognize)`

If each stage has 90% accuracy: **0.90\u00b3 = 0.729 (72.9% overall)**

---

### Pipeline 2: Donut (End-to-End Transformer)

```
 Receipt Image
      \u2502
      \u25bc
 \u250c\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2510
 \u2502  Swin Transformer Encoder         \u2502 \u2190 "Reads" pixels directly
 \u2502  (visual feature extraction)       \u2502
 \u2514\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2518
      \u2502  cross-attention
      \u25bc
 \u250c\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2510
 \u2502  BART Decoder                     \u2502 \u2190 Generates structured output
 \u2502  (autoregressive text generation)  \u2502    token by token
 \u2514\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2518
      \u2502
      \u25bc
 <s_company>VENDOR</s_company><s_date>DATE</s_date><s_total>TOTAL</s_total>
```

**Single differentiable pipeline:** one model, one loss function, no cascading errors.

---

### Key Architectural Differences

| Aspect | OCR Pipeline | Donut |
|--------|-------------|-------|
| **Stages** | 3 independent | 1 unified |
| **Error propagation** | Cascading (multiplicative) | Single-point |
| **Text representation** | Explicit strings | Implicit (learned embeddings) |
| **Layout understanding** | None (sequential text only) | Implicit (Swin attends to spatial regions) |
| **Training signal** | Each stage trained separately | End-to-end on final task |
| **Failure mode** | Brittle: one bad stage ruins output | Graceful: decoder compensates |

In [ ]:
# =============================================================================
# Cell 3 --- Why Each Field Behaves Differently
# =============================================================================
from IPython.display import display, Markdown

analysis_table = """
## Why Each Field Behaves Differently

| Aspect | Vendor (Donut wins) | Date (OCR wins) | Total (Donut wins) |
|--------|:---:|:---:|:---:|
| **Our F1: OCR / Donut-FT** | 0.49 / **0.82** | **0.78** / 0.63 | 0.63 / **0.78** |
| **Vocabulary** | Open (infinite vendor names) | Closed (date formats) | Semi-closed (numeric) |
| **Pattern regularity** | Low (diverse fonts, logos) | High (DD/MM/YYYY) | Medium (XX.XX) |
| **OCR challenge** | Must pick correct line from many; address/phone confusion | Digits are OCR-friendly; regex covers 95% of formats | Must distinguish TOTAL from SUBTOTAL, TAX, phone numbers |
| **Donut advantage** | Learns spatial prior: company = header region | Can handle unusual formats | Semantic context: total is near bottom, after items |
| **Why the winner wins** | Rule-based "first line" heuristic fails on diverse layouts | Regex patterns are near-perfect for standard date formats | Donut avoids order-of-magnitude phone number errors |
| **Dominant OCR error** | `address_confused` | `empty_prediction` (missed) | `order_of_magnitude` |
| **Dominant Donut error** | `wrong_entity` (different company) | `wrong_year` (hallucination) | `wrong_amount` (close but not exact) |

### Theoretical Explanation

**Vendor (open vocabulary problem):**
Company names form an essentially infinite set with no predefined patterns.
OCR can only use heuristics ("take the first non-skip line"), which fails
when addresses or phone numbers appear before the company name.
Donut learns from training data that the company name typically occupies
the header region and follows certain visual patterns (larger font, centered).

**Date (closed vocabulary advantage):**
Dates follow a small number of formats (DD/MM/YYYY, YYYY-MM-DD, etc.).
Regular expressions can capture these patterns with near-perfect precision.
When OCR correctly reads the digits, regex extraction is exact.
Donut's decoder, being autoregressive, can hallucinate plausible but wrong
years (e.g., 2018 instead of 2019) due to pre-training distribution biases.

**Total (semantic context matters):**
The total amount requires distinguishing the final sum from subtotals,
taxes, quantities, and other numbers. This is fundamentally a semantic task.
OCR + keyword matching works when "TOTAL" appears clearly, but fails
when the keyword is garbled or absent. Donut understands the document
structure: the total is typically the largest number near the bottom,
associated with a summary section.
"""
display(Markdown(analysis_table))

In [ ]:
# =============================================================================
# Cell 4 --- Error Propagation Framework (visual)
# =============================================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Left: OCR cascading error model ---
ax = axes[0]
stages = ['Detection\n(CRAFT)', 'Recognition\n(CRNN)', 'Extraction\n(Rules)']
acc_values = [0.95, 0.90, 0.85]  # hypothetical per-stage accuracy
cumulative = [acc_values[0]]
for a in acc_values[1:]:
    cumulative.append(cumulative[-1] * a)

x = np.arange(len(stages))
# Per-stage bars
ax.bar(x - 0.18, acc_values, 0.35, label='Per-stage accuracy',
       color='#4c78a8', edgecolor='black', linewidth=0.5)
# Cumulative line
ax.plot(x, cumulative, 'o-', color='#d62728', linewidth=2.5, markersize=8,
        label='Cumulative (cascading)', zorder=5)

for i, (a, c) in enumerate(zip(acc_values, cumulative)):
    ax.text(i - 0.18, a + 0.02, f'{a:.0%}', ha='center', fontsize=9, color='#4c78a8')
    ax.text(i, c - 0.05, f'{c:.1%}', ha='center', fontsize=10, color='#d62728', fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(stages)
ax.set_ylabel('Accuracy')
ax.set_title('OCR Pipeline: Cascading Error Propagation')
ax.set_ylim(0, 1.15)
ax.legend(loc='upper right', fontsize=9)
ax.grid(axis='y', alpha=0.2)

# --- Right: End-to-end vs cascading comparison ---
ax = axes[1]
scenarios = ['Ideal\n(each 95%)', 'Moderate\n(each 90%)', 'Degraded\n(each 85%)']
per_stage = [0.95, 0.90, 0.85]
cascade_3 = [p**3 for p in per_stage]  # OCR: 3 stages
single = per_stage  # Donut: equivalent single-stage

x = np.arange(len(scenarios))
ax.bar(x - 0.18, cascade_3, 0.35, label='OCR (3-stage cascade)',
       color='#4c78a8', edgecolor='black', linewidth=0.5)
ax.bar(x + 0.18, single, 0.35, label='Donut (single stage)',
       color='#f58518', edgecolor='black', linewidth=0.5)

for i, (c, s) in enumerate(zip(cascade_3, single)):
    ax.text(i - 0.18, c + 0.02, f'{c:.1%}', ha='center', fontsize=9)
    ax.text(i + 0.18, s + 0.02, f'{s:.0%}', ha='center', fontsize=9)
    # Gap annotation
    gap = s - c
    ax.annotate(f'+{gap:.1%}', xy=(i + 0.18, s),
                xytext=(i + 0.45, s - 0.05),
                fontsize=8, color='#e74c3c', fontweight='bold',
                arrowprops=dict(arrowstyle='->', color='#e74c3c', lw=1.2))

ax.set_xticks(x)
ax.set_xticklabels(scenarios)
ax.set_ylabel('Overall Accuracy')
ax.set_title('Cascading Errors: OCR vs End-to-End')
ax.set_ylim(0, 1.15)
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.2)

plt.suptitle('Error Propagation Framework', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_ROOT / 'fig_error_propagation_quick.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_error_propagation_quick.png')

In [ ]:
# =============================================================================
# Cell 5 --- Gap Analysis: Why our scores < published SOTA
# =============================================================================
display(Markdown("""
## Gap Analysis: Our Results vs Published SOTA

| Factor | Impact | Our Setup | Published SOTA |
|--------|:------:|-----------|----------------|
| **Training data** | \u2b50\u2b50\u2b50 | 240 docs (quick mode) | 626 docs (full) |
| **Validation set** | \u2b50\u2b50\u2b50 | 80 docs (23% of test) | 347 docs (full) |
| **Training epochs** | \u2b50\u2b50\u2b50 | 2 epochs | 30+ epochs |
| **Layout features** | \u2b50\u2b50 | None (text-only OCR, image-only Donut) | Text + 2D bounding boxes + image |
| **OCR extractor** | \u2b50\u2b50 | Simple regex rules | Learned NER / token classification |
| **Resolution** | \u2b50 | 1280\u00d7960 (reduced for T4) | 2560\u00d71920 (original) |
| **Ensemble** | \u2b50 | Single model | Often multi-model ensemble |
| **Post-processing** | \u2b50 | Minimal | Dictionary lookup, fuzzy matching |

### What this means:

1. **Quick mode** is the largest factor. With 80 val docs and 2 epochs, our
   results are preliminary estimates, not final benchmarks. Published Donut
   achieves **94.4% F1** with full training vs our **74.9%**.

2. **Layout features** add 2-3 F1 points consistently across benchmarks
   (BERT 93.7 \u2192 LayoutLM 95.1 \u2192 BROS 96.6). We do not use them.

3. **The pedagogical value** of this project is in comparing paradigms
   (OCR vs end-to-end) and understanding WHY they differ, not in achieving SOTA.
"""))

In [ ]:
# =============================================================================
# Cell 6 --- Robustness: Theoretical Explanation
# =============================================================================
display(Markdown("""
## Why Donut-FT is More Robust to Image Corruption

Our experiment shows:
- **OCR** average F1 degradation: **-0.166** (16.6%)
- **Donut-FT** average F1 degradation: **-0.119** (11.9%)

### Theoretical Explanation

**1. Error propagation chain (OCR):**

Corruption affects each OCR stage independently:
- **Detection (CRAFT):** Blur + downscale = weaker text boundaries \u2192 missed regions
- **Recognition (CRNN):** JPEG artifacts = character confusion (O\u2192C, 1\u2192l) \u2192 garbled text
- **Extraction (Rules):** Garbled text = keyword failure ("TCTAL" \u2260 "TOTAL") \u2192 wrong fields

Each stage fails partially, and errors multiply:
`P(correct|corrupt) = P(detect|corrupt) \u00d7 P(recognize|corrupt) \u00d7 P(extract|corrupt)`

**2. Single-stage resilience (Donut):**

- Swin Transformer encoder is inherently robust to low-frequency noise
  (blur, downscale are low-frequency; high-frequency JPEG artifacts
  are partially absorbed by the encoder's patch-based architecture)
- The BART decoder acts as a **language model**: it can "fill in" partially
  corrupted regions using its learned prior about receipt structure
- No explicit text detection = no cascade

**3. Practical implication:**

In real-world deployment (mobile scanning, messenger forwarding, faxes),
image quality is unpredictable. End-to-end models provide more consistent
performance, making them preferable for production systems where reliability
matters more than peak clean-image accuracy.
"""))

In [ ]:
# =============================================================================
# Cell 7 --- The Role of Fine-Tuning
# =============================================================================
display(Markdown("""
## Why Fine-Tuning is the Biggest Lever for Donut

Our results confirm the pattern from published literature:

| Model | F1 (our) | F1 (published) |
|-------|:--------:|:--------------:|
| Donut pre-trained (zero-shot) | **0.02** | ~84% |
| Donut fine-tuned on SROIE | **0.75** | ~94% |
| Improvement | **\u00d737.5** | **+10 pts** |

### Why does pre-trained Donut fail so badly?

1. **Output format mismatch:** The pre-trained model was trained on SynthDoG
   (synthetic documents) with a different output schema. It doesn't know
   SROIE-specific tags (`<s_company>`, `<s_date>`, `<s_total>`).

2. **Domain gap:** Synthetic documents \u2260 real Malaysian receipts. Different
   fonts, layouts, thermal paper artifacts, multilingual text (Malay/English).

3. **Decoder vocabulary:** Without fine-tuning, the decoder doesn't have
   special tokens for SROIE fields. It produces free-form text that
   doesn't match the expected structured output.

### What fine-tuning teaches:

- **Encoder:** Focus on receipt-specific visual features (header region for
  company, bottom area for totals, date clusters)
- **Decoder:** Generate the exact JSON-like schema SROIE expects
- **Cross-attention:** Map specific image regions to specific output fields

This is consistent with the general finding that **domain-specific fine-tuning
is essential for Document AI models** (Kim et al., 2022; Xu et al., 2020).
"""))

In [ ]:
# =============================================================================
# Cell 8 --- References
# =============================================================================
display(Markdown("""
## References

1. **Huang, Z., Chen, K., He, J., et al.** (2019). "ICDAR2019 Competition on
   Scanned Receipt OCR and Information Extraction." *ICDAR 2019.*
   [arXiv:2103.10213](https://arxiv.org/abs/2103.10213)

2. **Kim, G., Hong, T., Yim, M., et al.** (2022). "OCR-free Document
   Understanding Transformer (Donut)." *ECCV 2022.*
   [arXiv:2111.15664](https://arxiv.org/abs/2111.15664)

3. **Xu, Y., Li, M., Cui, L., et al.** (2020). "LayoutLM: Pre-training of
   Text and Layout for Document Image Understanding." *KDD 2020.*
   [arXiv:1912.13318](https://arxiv.org/abs/1912.13318)

4. **Xu, Y., Xu, Y., Lv, T., et al.** (2021). "LayoutLMv2: Multi-modal
   Pre-training for Visually-Rich Document Understanding." *ACL 2021.*
   [arXiv:2012.14740](https://arxiv.org/abs/2012.14740)

5. **Huang, Y., Lv, T., Cui, L., et al.** (2022). "LayoutLMv3: Pre-training
   for Document AI with Unified Text and Image Masking." *ACM MM 2022.*
   [arXiv:2204.08387](https://arxiv.org/abs/2204.08387)

6. **Hong, T., Kim, D., Ji, M., et al.** (2022). "BROS: A Pre-trained
   Language Model Focusing on Text and Layout for Better Key Information
   Extraction from Documents." *AAAI 2022.*
   [arXiv:2108.04539](https://arxiv.org/abs/2108.04539)

7. **Yu, W., Lu, N., Qi, X., et al.** (2021). "PICK: Processing Key
   Information Extraction from Documents using Improved Graph
   Learning-Convolutional Networks." *ICPR 2020.*

8. **Schmid, P.** (2022). "Fine-tuning Donut for document parsing."
   [philschmid.de](https://www.philschmid.de/fine-tuning-donut)
"""))